# Advanced Modeling with Weather Features

## Section 1 — Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession

from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression


Importing plotly failed. Interactive plots will not work.


## Section 2 — Load Data

In [2]:
spark = SparkSession.builder.appName("Forecasting").getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "America/Toronto")

incWeatherFeatures_df = spark.read.parquet("../../data/weather/features_incWeather")

incWeatherFeatures_df.printSchema()
incWeatherFeatures_df.show(5)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/03 17:01:26 WARN Utils: Your hostname, users-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.4.39 instead (on interface en0)
26/04/03 17:01:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/03 17:01:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- station_id: string (nullable = true)
 |-- ts_hour: timestamp (nullable = true)
 |-- demand_count: long (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- is_holiday: integer (nullable = true)
 |-- community_id: long (nullable = true)
 |-- temp: float (nullable = true)
 |-- precip: float (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- metro_proximity: string (nullable = true)
 |-- metro_name: string (nullable = true)
 |-- dist_km: double (nullable = true)
 |-- transit_proximity_tier: string (nullable = true)



+--------------------+-------------------+------------+-----------+----------+----------+------------+-----+------+---------+----------+---------------+-------------+-------------------+----------------------+
|          station_id|            ts_hour|demand_count|day_of_week|is_weekend|is_holiday|community_id| temp|precip|      lat|       lon|metro_proximity|   metro_name|            dist_km|transit_proximity_tier|
+--------------------+-------------------+------------+-----------+----------+----------+------------+-----+------+---------+----------+---------------+-------------+-------------------+----------------------+
|45.524790,-73.565...|2024-06-24 16:00:00|           9|          2|         0|         1|        NULL| 21.4|   0.0| 45.52479| -73.56545|           High|   Sherbrooke| 0.7256533163973264|  Tier 2: Commuter ...|
|45.506539,-73.576...|2024-10-19 18:00:00|          18|          7|         1|         0|        NULL| 13.9|   0.0|45.506539|-73.576609|           High|       M

## Section 3 — Convert to Pandas

In [3]:
incWeatherFeatures_df.describe().show()

26/04/03 16:22:10 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+--------------------+-----------------+-----------------+-------------------+-------------------+------------+------------------+-------------------+-------------------+------------------+---------------+-----------------+--------------------+----------------------+
|summary|          station_id|     demand_count|      day_of_week|         is_weekend|         is_holiday|community_id|              temp|             precip|                lat|               lon|metro_proximity|       metro_name|             dist_km|transit_proximity_tier|
+-------+--------------------+-----------------+-----------------+-------------------+-------------------+------------+------------------+-------------------+-------------------+------------------+---------------+-----------------+--------------------+----------------------+
|  count|             6179615|          6179615|          6179615|            6179615|            6179615|           0|           6141286|            6141286|            61

In [3]:
pdf = incWeatherFeatures_df.toPandas()

pdf['date'] = pd.to_datetime(pdf['date'])
pdf = pdf.sort_values('date')

pdf.head()


[57.671s][warning][gc,alloc] Executor task launch worker for task 6.0 in stage 5.0 (TID 16): Retried waiting for GCLocker too often allocating 131074 words


26/04/03 16:22:53 ERROR Executor: Exception in task 6.0 in stage 5.0 (TID 16)
java.lang.OutOfMemoryError: Java heap space
	at java.base/java.nio.HeapByteBuffer.<init>(HeapByteBuffer.java:71)
	at java.base/java.nio.ByteBuffer.allocate(ByteBuffer.java:391)
	at org.apache.spark.serializer.SerializerHelper$.$anonfun$serializeToChunkedBuffer$1(SerializerHelper.scala:40)
	at org.apache.spark.serializer.SerializerHelper$.$anonfun$serializeToChunkedBuffer$1$adapted(SerializerHelper.scala:40)
	at org.apache.spark.serializer.SerializerHelper$$$Lambda/0x000000012bd0d9c8.apply(Unknown Source)
	at org.apache.spark.util.io.ChunkedByteBufferOutputStream.allocateNewChunkIfNeeded(ChunkedByteBufferOutputStream.scala:87)
	at org.apache.spark.util.io.ChunkedByteBufferOutputStream.write(ChunkedByteBufferOutputStream.scala:75)
	at java.base/java.io.ObjectOutputStream$BlockDataOutputStream.write(ObjectOutputStream.java:1875)
	at java.base/java.io.ObjectOutputStream.write(ObjectOutputStream.java:725)
	at org.

ConnectionRefusedError: [Errno 61] Connection refused

ConnectionRefusedError: [Errno 61] Connection refused

In [ ]:
# Sort by date
from pyspark.sql import functions as F

incWeatherFeatures_df = incWeatherFeatures_df.withColumn(
    "ts_hour",
    F.to_timestamp("ts_hour")
)
# incWeatherFeatures_df = incWeatherFeatures_df.orderBy("ts_hour")

from pyspark.sql.window import Window
# window = Window.orderBy("ts_hour")
windowSpec = Window.partitionBy("station_id").orderBy("ts_hour")

incWeatherFeatures_df = incWeatherFeatures_df.withColumn("row_num", F.row_number().over(windowSpec))

total_rows = incWeatherFeatures_df.count()
split_point = int(total_rows * 0.8)

train_incWeatherFeatures_df = incWeatherFeatures_df.filter(F.col("row_num") <= split_point)
test_incWeatherFeatures_df  = incWeatherFeatures_df.filter(F.col("row_num") > split_point)

## Section 4 — Train/Test Split

In [ ]:
# Separate FEATURES vs LABEL
label_col = "demand_count"

# extract hour and day features from ts_hour
train_incWeatherFeatures_df = train_incWeatherFeatures_df.withColumn("hour", F.hour("ts_hour")) \
       .withColumn("day", F.dayofmonth("ts_hour"))

# encode categorical features
from pyspark.ml.feature import StringIndexer
categorical_cols = [
    'station_id',
    'metro_proximity',
    'metro_name',
    'transit_proximity_tier'
]
indexers = [
    StringIndexer(inputCol=col, outputCol=col+"_idx", handleInvalid="keep")
    for col in categorical_cols
]

from pyspark.ml.feature import VectorAssembler
# ensure Final Feature Columns (ONLY NUMERIC)
feature_cols = [
    'day_of_week', 'is_weekend', 'is_holiday',
    'community_id', 'temp', 'precip',
    'lat', 'lon', 'dist_km',
    'hour', 'day'
] + [col + "_idx" for col in categorical_cols]


assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"   # ✅ FIXED
)

In [ ]:
from pyspark.ml import Pipeline

# Apply window operations BEFORE the pipeline if they are custom SQL-like features
# (Assuming you need some rolling weather features)
train_incWeatherFeatures_df = train_incWeatherFeatures_df.withColumn(
    "temp_lag", F.lag("temp").over(windowSpec)
).fillna(0)

# Re-run Pipeline
pipeline = Pipeline(stages=indexers + [assembler])
pipeline_model = pipeline.fit(train_incWeatherFeatures_df)

train_df = pipeline_model.transform(train_incWeatherFeatures_df)
test_df  = pipeline_model.transform(test_incWeatherFeatures_df)

26/04/03 16:48:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/03 16:48:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/03 16:48:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/03 16:48:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/03 16:48:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/03 16:48:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/03 1

[1173.979s][warning][gc,alloc] Executor task launch worker for task 0.0 in stage 81.0 (TID 270): Retried waiting for GCLocker too often allocating 16777218 words


26/04/03 16:49:27 WARN TaskMemoryManager: Failed to allocate a page (134217728 bytes), try again.
26/04/03 16:49:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/03 16:49:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/03 16:49:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/03 16:49:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/03 16:49:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/03 16:49:36 WARN WindowExec: No Partition Defined for Window operation!

IllegalArgumentException: [FIELD_NOT_FOUND] No such struct field `hour` in `station_id`, `ts_hour`, `demand_count`, `day_of_week`, `is_weekend`, `is_holiday`, `community_id`, `temp`, `precip`, `lat`, `lon`, `metro_proximity`, `metro_name`, `dist_km`, `transit_proximity_tier`, `row_num`, `station_id_idx`, `metro_proximity_idx`, `metro_name_idx`, `transit_proximity_tier_idx`. SQLSTATE: 42704

## Section 5 — Prophet Models

In [ ]:
def train_prophet(df, target='outflow'):
    prophet_df = df[['date', target]].rename(columns={'date': 'ds', target: 'y'})

    model = Prophet(weekly_seasonality=True, daily_seasonality=True)
    model.fit(prophet_df)

    return model

def train_weather_prophet(df, target='outflow'):
    df = df.copy()
    prophet_df = df[['date', target]].rename(columns={'date': 'ds', target: 'y'})

    extra_features = ['temperature', 'humidity', 'rainfall', 'hour_sin', 'hour_cos', 'day_of_week']

    model = Prophet()

    for col in extra_features:
        if col in df.columns:
            model.add_regressor(col)
            prophet_df[col] = df[col]

    model.fit(prophet_df)
    return model

def predict_prophet(model, df):
    future = df.rename(columns={'date': 'ds'})
    forecast = model.predict(future)
    return forecast


## Section 6 — XGBoost

In [ ]:
def get_features(df):
    return [col for col in df.columns if col not in ['date', 'outflow']]

def train_xgboost(train_df, test_df, target='outflow'):
    features = get_features(train_df)

    model = xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6)
    model.fit(train_df[features], train_df[target])

    preds = model.predict(test_df[features])

    print("MAE:", mean_absolute_error(test_df[target], preds))
    print("RMSE:", np.sqrt(mean_squared_error(test_df[target], preds)))

    return model, preds


## Section 7 — Additional Models

In [ ]:
def train_random_forest(train_df, test_df, target='outflow'):
    features = get_features(train_df)

    model = RandomForestRegressor(n_estimators=100)
    model.fit(train_df[features], train_df[target])

    preds = model.predict(test_df[features])
    print("RF MAE:", mean_absolute_error(test_df[target], preds))

    return model, preds

def train_linear_regression(train_df, test_df, target='outflow'):
    features = get_features(train_df)

    model = LinearRegression()
    model.fit(train_df[features], train_df[target])

    preds = model.predict(test_df[features])
    print("LR MAE:", mean_absolute_error(test_df[target], preds))

    return model, preds


## Section 8 — Run Models

In [ ]:
prophet_model = train_prophet(train_df)
prophet_forecast = predict_prophet(prophet_model, test_df)

weather_prophet_model = train_weather_prophet(train_df)
weather_forecast = predict_prophet(weather_prophet_model, test_df)

xgb_model, xgb_preds = train_xgboost(train_df, test_df)

rf_model, rf_preds = train_random_forest(train_df, test_df)

lr_model, lr_preds = train_linear_regression(train_df, test_df)


## Section 9 — Visualization

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(test_df['date'], test_df['outflow'], label='Actual')
plt.plot(test_df['date'], xgb_preds, label='XGBoost')
plt.plot(test_df['date'], rf_preds, label='RandomForest')

plt.legend()
plt.title("Model Comparison")
plt.show()
